# Healthcare Insurance Cost Prediction

This notebook walks through the complete process of building a Linear Regression model to predict healthcare insurance costs.

**Steps:**
1. Load and explore the dataset
2. Visualize key relationships
3. Encode categorical features
4. Train a Linear Regression model
5. Evaluate model performance
6. Save the trained model

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Load Dataset

In [ ]:
df = pd.read_csv("data/v1/raw/insurance.csv")
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(10)

## 3. Dataset Info and Summary Statistics

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

## 4. Exploratory Data Analysis

### 4.1 Distribution of Insurance Charges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["charges"], bins=40, color="steelblue", edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Insurance Charges ($)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Insurance Charges")

axes[1].boxplot(df["charges"], vert=True)
axes[1].set_ylabel("Insurance Charges ($)")
axes[1].set_title("Boxplot of Insurance Charges")

plt.tight_layout()
plt.show()

print(f"Mean:   ${df['charges'].mean():,.2f}")
print(f"Median: ${df['charges'].median():,.2f}")
print(f"Min:    ${df['charges'].min():,.2f}")
print(f"Max:    ${df['charges'].max():,.2f}")

### 4.2 Smoker vs Non-Smoker Costs

In [ ]:
smoker_avg = df.groupby("smoker")["charges"].mean()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(smoker_avg.index, smoker_avg.values, color=["#2ecc71", "#e74c3c"], edgecolor="black", alpha=0.8)
ax.set_xlabel("Smoker Status")
ax.set_ylabel("Average Charges ($)")
ax.set_title("Average Insurance Charges: Smoker vs Non-Smoker")

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 300,
            f"${height:,.0f}", ha="center", fontweight="bold", fontsize=12)

plt.tight_layout()
plt.show()

diff = smoker_avg["yes"] - smoker_avg["no"]
print(f"Smokers pay ${diff:,.0f} more on average.")

### 4.3 Age vs Charges

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = df["smoker"].map({"yes": "#e74c3c", "no": "#2ecc71"})
ax.scatter(df["age"], df["charges"], c=colors, alpha=0.5, edgecolors="black", linewidth=0.3)
ax.set_xlabel("Age")
ax.set_ylabel("Charges ($)")
ax.set_title("Age vs Insurance Charges (colored by smoker status)")

import matplotlib.patches as mpatches
legend_elements = [
    mpatches.Patch(color="#e74c3c", label="Smoker"),
    mpatches.Patch(color="#2ecc71", label="Non-Smoker")
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

### 4.4 BMI vs Charges

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = df["smoker"].map({"yes": "#e74c3c", "no": "#2ecc71"})
ax.scatter(df["bmi"], df["charges"], c=colors, alpha=0.5, edgecolors="black", linewidth=0.3)
ax.set_xlabel("BMI")
ax.set_ylabel("Charges ($)")
ax.set_title("BMI vs Insurance Charges (colored by smoker status)")

legend_elements = [
    mpatches.Patch(color="#e74c3c", label="Smoker"),
    mpatches.Patch(color="#2ecc71", label="Non-Smoker")
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

### 4.5 Charges by Region

In [ ]:
region_avg = df.groupby("region")["charges"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(region_avg.index, region_avg.values, color="#3498db", edgecolor="black", alpha=0.8)
ax.set_xlabel("Region")
ax.set_ylabel("Average Charges ($)")
ax.set_title("Average Insurance Charges by Region")

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 200,
            f"${height:,.0f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

### 4.6 Correlation Heatmap

In [ ]:
# Encode for correlation analysis
df_encoded = df.copy()
df_encoded["sex"] = df_encoded["sex"].map({"male": 1, "female": 0})
df_encoded["smoker"] = df_encoded["smoker"].map({"yes": 1, "no": 0})
df_encoded["region"] = df_encoded["region"].map({"northeast": 0, "northwest": 1, "southeast": 2, "southwest": 3})

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_encoded.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=1, ax=ax)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

print("\nCorrelation with charges:")
print(corr["charges"].sort_values(ascending=False))

### 4.7 Categorical Feature Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sex distribution
df["sex"].value_counts().plot(kind="bar", ax=axes[0], color=["#3498db", "#e91e63"], edgecolor="black", alpha=0.8)
axes[0].set_title("Gender Distribution")
axes[0].set_xlabel("")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

# Smoker distribution
df["smoker"].value_counts().plot(kind="bar", ax=axes[1], color=["#2ecc71", "#e74c3c"], edgecolor="black", alpha=0.8)
axes[1].set_title("Smoker Distribution")
axes[1].set_xlabel("")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)

# Region distribution
df["region"].value_counts().plot(kind="bar", ax=axes[2], color="#9b59b6", edgecolor="black", alpha=0.8)
axes[2].set_title("Region Distribution")
axes[2].set_xlabel("")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 5. Data Preprocessing

In [ ]:
# Encode categorical columns
df["sex"] = df["sex"].map({"male": 1, "female": 0})
df["smoker"] = df["smoker"].map({"yes": 1, "no": 0})
df["region"] = df["region"].map({
    "northeast": 0,
    "northwest": 1,
    "southeast": 2,
    "southwest": 3
})

print("Encoded dataset:")
df.head()

In [ ]:
# Split features and target
X = df.drop("charges", axis=1)
y = df["charges"]

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

In [ ]:
# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

## 6. Model Training

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully.")
print(f"\nIntercept: {model.intercept_:,.2f}")
print(f"\nCoefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature:>10}: {coef:>10,.2f}")

## 7. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Model Performance on Test Set")
print("=" * 40)
print(f"R2 Score : {r2:.4f} ({r2*100:.2f}%)")
print(f"MAE      : ${mae:,.2f}")
print(f"RMSE     : ${rmse:,.2f}")

In [ ]:
# Actual vs Predicted scatter plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred, alpha=0.5, color="steelblue", edgecolors="black", linewidth=0.3)

# Perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2, label="Perfect Prediction")

ax.set_xlabel("Actual Charges ($)")
ax.set_ylabel("Predicted Charges ($)")
ax.set_title(f"Actual vs Predicted Insurance Charges (R2 = {r2:.4f})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residuals distribution
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=30, color="coral", edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Residual ($)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Residuals")
axes[0].axvline(x=0, color="black", linestyle="--", linewidth=1)

axes[1].scatter(y_pred, residuals, alpha=0.5, color="coral", edgecolors="black", linewidth=0.3)
axes[1].axhline(y=0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Predicted Charges ($)")
axes[1].set_ylabel("Residual ($)")
axes[1].set_title("Residuals vs Predicted Values")

plt.tight_layout()
plt.show()

print(f"Mean residual:   ${residuals.mean():,.2f}")
print(f"Std of residuals: ${residuals.std():,.2f}")

## 8. Feature Importance

In [ ]:
importance = pd.Series(np.abs(model.coef_), index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind="barh", color="#3498db", edgecolor="black", alpha=0.8, ax=ax)
ax.set_xlabel("Absolute Coefficient Value")
ax.set_title("Feature Importance (Linear Regression Coefficients)")
plt.tight_layout()
plt.show()

## 9. Save Model

In [ ]:
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved as model.pkl")

## 10. Test Prediction

In [ ]:
# Load saved model and make a sample prediction
with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# Sample input: 30 year old male, BMI 28, 2 children, non-smoker, northwest
sample = np.array([[30, 1, 28.0, 2, 0, 1]])
prediction = loaded_model.predict(sample)[0]

print(f"Sample prediction: ${prediction:,.2f}")